In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

from langchain_openai import ChatOpenAI
from langchain_core.messages import RemoveMessage

from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("Grok_Api_key"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
def chat_model(state: MessagesState):
    response = llm.invoke(state['messages'])
    return {'messages':['response']}


def del_old_mess(state:MessagesState):
    msgs = state['messages']
    
    if len(msgs) > 10:
        to_remove = msgs[:6]

        return {'messages': [RemoveMessage(id=m.id)  for m in to_remove]}

    return {}

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("chat", chat_model)
builder.add_node("cleanup", del_old_mess)

In [ ]:

builder.add_edge(START, "chat")
builder.add_edge("chat", "cleanup")   # run deletion after each response
builder.add_edge("cleanup", "__end__")

In [ ]:
graph = builder.compile(checkpointer=InMemorySaver())

In [ ]:
graph

In [ ]:
config = {"configurable": {"thread_id": "t1"}}


# Run multiple turns
graph.invoke({"messages": [{"role": "user", "content": "Hi, I'm Nitish"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Tell me about LangGraph"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Now explain checkpointers"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Langchain"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Quantum Mechanics"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Gen AI"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is my name"}]}, config)

In [ ]:
snap = graph.get_state(config)

print('stored messages after cleanup', len(snap.vaues['messages']))